In [15]:
import pandas as pd
import geopandas as gpd
import pyarrow.parquet as pq
import numpy as np
import duckdb
import requests, zipfile, io, os
import matplotlib.pyplot as plt

In [16]:
SHORE_COUNTY_FIPS = {"001", "009", "025", "029"}  # Atlantic, Cape May, Monmouth, Ocean
CUTOFF_YEAR = 2013  # mod_iv_year <= 2013 → ACS 2013;  >= 2014 → ACS 2023

In [18]:
acs_2013 = pd.read_csv("data/acs_seasonal_early.csv", dtype={"GEOID": str})
acs_2023 = pd.read_csv("data/acs_seasonal_late.csv", dtype={"GEOID": str})

print(f"Tracts 2013: {len(acs_2013):,}  |  Tracts 2023: {len(acs_2023):,}")
acs_2013.head(3)

Tracts 2013: 373  |  Tracts 2023: 407


,GEOID,NAME,totalE,totalM,seasonalE,seasonalM,pct_seasonal,period
0,34009020101,"Census Tract 201.01, Cape May County, New Jersey",4881,174,2828,226,0.579389,early
1,34009020302,"Census Tract 203.02, Cape May County, New Jersey",2724,122,747,146,0.274229,early
2,34009021400,"Census Tract 214, Cape May County, New Jersey",5813,277,3787,284,0.651471,early


In [ ]:
# Download NJ Census TIGER tract boundaries (2023 vintage, cached locally)
TIGER_DIR = "data/tiger_tracts/"
TIGER_SHP = TIGER_DIR + "tl_2023_34_tract.shp"

if not os.path.exists(TIGER_SHP):
    os.makedirs(TIGER_DIR, exist_ok=True)
    r = requests.get("https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_34_tract.zip")
    r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
        zf.extractall(TIGER_DIR)

tracts = gpd.read_file(TIGER_SHP)
tracts = tracts[tracts["COUNTYFP"].isin(SHORE_COUNTY_FIPS)][["GEOID", "geometry"]].copy()
print(f"Shore census tracts: {len(tracts)}")
tracts.head(3)

Shore census tracts: 407


,GEOID,geometry
103,34025800101,"POLYGON ((-74.00497 40.40531, -74.0049 40.4054..."
104,34025809503,"POLYGON ((-74.29084 40.37301, -74.29057 40.373..."
105,34025808901,"POLYGON ((-74.0539 40.14465, -74.05384 40.1448..."


In [ ]:
# Build a stable gis_pin → GEOID lookup via centroid spatial join.
# Use one year of parcel geometries; gis_pin is stable across years.
parcels_geo = gpd.read_parquet(
    "data/parcels_shore.parquet",
    filters=[("mod_iv_year", "==", 2023)],
    columns=["gis_pin", "geometry"],
)

# Compute centroids in native CRS (EPSG:3424), reproject tracts to match
centroids = parcels_geo.copy()
centroids["geometry"] = parcels_geo.geometry.centroid
tracts_nj = tracts.to_crs(centroids.crs)

pin_geoid = (
    gpd.sjoin(centroids[["gis_pin", "geometry"]], tracts_nj, how="left", predicate="within")
    [["gis_pin", "GEOID"]]
    .drop_duplicates("gis_pin")
    .reset_index(drop=True)
)

matched = pin_geoid["GEOID"].notna().sum()
print(f"Unique parcels: {len(pin_geoid):,}  |  Matched to tract: {matched:,}  ({matched/len(pin_geoid):.1%})")
pin_geoid.head()

Unique parcels: 442,588  |  Matched to tract: 440,015  (99.4%)


,gis_pin,GEOID
0,0108_1124_14,34001011702
1,0108_1124_16,34001011702
2,0108_1124_17,34001011702
3,0108_1124_18,34001011702
4,0108_1124_20,34001011702


In [ ]:
# Load full parcel tabular data
parcels = pq.read_table(
    "data/parcels_shore.parquet",
    columns=[
        "gis_pin", "mod_iv_year", "mod_iv_county_name", "property_class",
        "property_class_code_name", "sale_price", "land_value",
        "improvement_value", "net_taxable_value", "year_constructed",
        "property_location",
    ],
).to_pandas()

# Attach tract GEOID
parcels = parcels.merge(pin_geoid, on="gis_pin", how="left")

# Join both ACS vintages by GEOID
parcels = (
    parcels
    .merge(acs_2013[["GEOID", "pct_seasonal"]].rename(columns={"pct_seasonal": "_pct_2013"}), on="GEOID", how="left")
    .merge(acs_2023[["GEOID", "pct_seasonal"]].rename(columns={"pct_seasonal": "_pct_2023"}), on="GEOID", how="left")
)

# Single pct_seasonal: use 2013 ACS for earlier years, 2023 ACS for more recent years
parcels["pct_seasonal"] = np.where(
    parcels["mod_iv_year"] <= CUTOFF_YEAR,
    parcels["_pct_2013"],
    parcels["_pct_2023"],
)
parcels = parcels.drop(columns=["_pct_2013", "_pct_2023"])

print(f"Shape: {parcels.shape}")
parcels[["gis_pin", "mod_iv_year", "GEOID", "pct_seasonal"]].head()

In [ ]:
# --- Verification ---
n = len(parcels)
has_geoid = parcels["GEOID"].notna().sum()
has_pct   = parcels["pct_seasonal"].notna().sum()

print("=== Join Coverage ===")
print(f"Total rows (all years):    {n:>10,}")
print(f"With tract GEOID:          {has_geoid:>10,}  ({has_geoid/n:.1%})")
print(f"With pct_seasonal:         {has_pct:>10,}  ({has_pct/n:.1%})")
print()

print("=== Tract match rate by county ===")
print(
    parcels[parcels["mod_iv_year"] == 2023]
    .groupby("mod_iv_county_name")["GEOID"]
    .agg(total="count", matched=lambda x: x.notna().sum())
    .assign(pct=lambda d: (d["matched"] / d["total"]).map("{:.1%}".format))
    .to_string()
)
print()

tract_stats = parcels.drop_duplicates("GEOID")[["pct_seasonal"]].dropna()
print(f"=== pct_seasonal distribution ({len(tract_stats)} unique tracts) ===")
print(tract_stats.describe().round(3))

parcels["pct_seasonal"].plot.hist(bins=40, figsize=(8, 4), title="% Seasonal Units by Tract")
plt.xlabel("pct_seasonal")
plt.tight_layout()
plt.show()

In [ ]:
# Save lookups, then merge both new columns into the full parquet via DuckDB
lookup = (
    parcels[["gis_pin", "mod_iv_year", "pct_seasonal"]]
    .drop_duplicates(["gis_pin", "mod_iv_year"])
)
lookup.to_parquet("data/acs_seasonal_lookup.parquet", index=False)
pin_geoid.to_parquet("data/pin_geoid.parquet", index=False)

con = duckdb.connect()
con.execute("""
    COPY (
        SELECT p.*, g.GEOID AS census_tract_geoid, l.pct_seasonal
        FROM read_parquet('data/parcels_shore.parquet') AS p
        LEFT JOIN read_parquet('data/pin_geoid.parquet') AS g
            USING (gis_pin)
        LEFT JOIN read_parquet('data/acs_seasonal_lookup.parquet') AS l
            USING (gis_pin, mod_iv_year)
    ) TO 'data/parcels_shore_tmp.parquet' (FORMAT PARQUET)
""")
con.close()

os.replace("data/parcels_shore_tmp.parquet", "data/parcels_shore.parquet")
print("Done — census_tract_geoid and pct_seasonal written to parcels_shore.parquet")